In [1]:
# ==========================================
# EXPERIMENT 3
# CSE-CIC-IDS2018 EXTERNAL TESTING
# ==========================================

import os
import glob
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

print("Experiment 3 started.")

Experiment 3 started.


In [2]:
# ==========================================
# LOAD EXPERIMENT 2 MODELS
# ==========================================

EXP2_PATH = "exp_2_result"

rf_model = joblib.load(
    os.path.join(
        EXP2_PATH,
        "experiment2_random_forest.pkl"
    )
)

iso_model = joblib.load(
    os.path.join(
        EXP2_PATH,
        "experiment2_isolation_forest.pkl"
    )
)

iso_scaler = joblib.load(
    os.path.join(
        EXP2_PATH,
        "experiment2_isolation_scaler.pkl"
    )
)

train_medians = joblib.load(
    os.path.join(
        EXP2_PATH,
        "experiment2_train_medians.pkl"
    )
)

hybrid_config = joblib.load(
    os.path.join(
        EXP2_PATH,
        "experiment2_hybrid_config.pkl"
    )
)

RF_WEIGHT = hybrid_config["rf_weight"]
ISO_WEIGHT = hybrid_config["isolation_weight"]
BEST_THRESHOLD = hybrid_config["best_threshold"]
FEATURE_COLUMNS = hybrid_config["feature_columns"]

print("Models loaded successfully.")

print("\nRF classes:")
print(rf_model.classes_)

print("\nRF weight:", RF_WEIGHT)
print("Isolation weight:", ISO_WEIGHT)
print("Threshold:", BEST_THRESHOLD)

print("\nFeature count:", len(FEATURE_COLUMNS))

Models loaded successfully.

RF classes:
[0 1]

RF weight: 0.5
Isolation weight: 0.5
Threshold: 0.515684507056306

Feature count: 67


In [3]:
# ==========================================
# CSE-CIC-IDS2018 DATA PATH
# ==========================================

DATA_2018_PATH = r"data_2018"

files_2018 = sorted(
    glob.glob(
        os.path.join(
            DATA_2018_PATH,
            "*.parquet"
        )
    )
)

print("Total 2018 files:", len(files_2018))

for f in files_2018:
    print(os.path.basename(f))

Total 2018 files: 10
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


In [4]:
# ==========================================
# LOAD CSE-CIC-IDS2018
# ==========================================

file_dfs_2018 = {}

for file_path in files_2018:

    filename = os.path.basename(file_path)

    df_temp = pd.read_parquet(file_path)

    file_dfs_2018[filename] = df_temp

    print(
        filename,
        "->",
        df_temp.shape
    )

print("\nCSE-CIC-IDS2018 loaded.")

Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet -> (771587, 78)
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet -> (619346, 78)
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet -> (954846, 78)
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet -> (561396, 78)
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet -> (794812, 78)
DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet -> (591873, 78)
Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet -> (456873, 78)
Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet -> (249170, 78)
Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet -> (830224, 78)
Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet -> (829405, 78)

CSE-CIC-IDS2018 loaded.


In [5]:
# ==========================================
# CHECK 2018 COLUMNS
# ==========================================

first_file = next(
    iter(file_dfs_2018.values())
)

print("2018 columns:")
print(first_file.columns.tolist())

print("\nTotal columns:", len(first_file.columns))

2018 columns:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [6]:
# ==========================================
# COMBINE ALL 2018 DATA
# ==========================================

df_2018 = pd.concat(
    file_dfs_2018.values(),
    ignore_index=True
)

print("Final 2018 shape:")
print(df_2018.shape)

print("\nColumns:")
print(len(df_2018.columns))

Final 2018 shape:
(6659532, 78)

Columns:
78


In [7]:
# ==========================================
# CHECK 2018 LABELS
# ==========================================

print("2018 Label distribution:")

print(
    df_2018["Label"]
    .astype(str)
    .str.strip()
    .value_counts()
)

2018 Label distribution:
Label
Benign                      5329008
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


In [8]:
# ==========================================
# CREATE 2018 BINARY TARGET
# ==========================================

df_2018["Target"] = (
    df_2018["Label"]
    .astype(str)
    .str.strip()
    .str.upper()
    .ne("BENIGN")
    .astype(int)
)

print("2018 Target distribution:")
print(
    df_2018["Target"].value_counts()
)

2018 Target distribution:
Target
0    5329008
1    1330524
Name: count, dtype: int64


In [9]:
# ==========================================
# SAME FEATURES AS 2017
# ==========================================

available_features = [
    col
    for col in FEATURE_COLUMNS
    if col in df_2018.columns
]

missing_features = [
    col
    for col in FEATURE_COLUMNS
    if col not in df_2018.columns
]

print("Required features:", len(FEATURE_COLUMNS))
print("Available features:", len(available_features))

print("\nMissing features:")
print(missing_features)

assert len(missing_features) == 0, (
    "2018 dataset is missing features required "
    "by the 2017 model."
)

X_2018 = df_2018[
    FEATURE_COLUMNS
].copy()

y_2018 = df_2018[
    "Target"
].copy()

print("\nX_2018:", X_2018.shape)
print("y_2018:", y_2018.shape)

Required features: 67
Available features: 67

Missing features:
[]

X_2018: (6659532, 67)
y_2018: (6659532,)


In [10]:
# ==========================================
# CLEAN 2018 FEATURES
# ==========================================

X_2018 = X_2018.replace(
    [np.inf, -np.inf],
    np.nan
)

# Use ONLY medians learned from 2017
for col in FEATURE_COLUMNS:

    if col in train_medians.index:
        X_2018[col] = X_2018[col].fillna(
            train_medians[col]
        )

print(
    "Remaining NaN:",
    X_2018.isna().sum().sum()
)

Remaining NaN: 0


In [11]:
# ==========================================
# RANDOM FOREST - 2018
# ==========================================

rf_2018_prob = rf_model.predict_proba(
    X_2018
)[:, 1]

rf_2018_pred = (
    rf_2018_prob >= BEST_THRESHOLD
).astype(int)

print("RF 2018 prediction completed.")

print(
    pd.Series(
        rf_2018_pred
    ).value_counts()
)

RF 2018 prediction completed.
0    6659260
1        272
Name: count, dtype: int64


In [14]:
# ==========================================
# ISOLATION FOREST - 2018
# ==========================================

X_2018_scaled = iso_scaler.transform(
    X_2018
)

iso_2018_raw = iso_model.decision_function(
    X_2018_scaled
)

# Higher = more anomalous
iso_2018_score = -iso_2018_raw

print("Isolation Forest 2018 score generated.")

Isolation Forest 2018 score generated.


In [15]:
# ==========================================
# NORMALIZE ISOLATION SCORE
# ==========================================

# Load Experiment 2 validation score range
# This must be recreated from saved model inputs,
# so for now use robust min/max from 2018 only
# for score inspection.

iso_2018_min = iso_2018_score.min()
iso_2018_max = iso_2018_score.max()

iso_2018_norm = (
    iso_2018_score - iso_2018_min
) / (
    iso_2018_max - iso_2018_min + 1e-12
)

print("Isolation score normalized.")

Isolation score normalized.


In [17]:
# ==========================================
# HYBRID RISK SCORE - 2018
# ==========================================

hybrid_2018_score = (
    RF_WEIGHT * rf_2018_prob
    +
    ISO_WEIGHT * iso_2018_norm
)

hybrid_2018_pred = (
    hybrid_2018_score >= BEST_THRESHOLD
).astype(int)

print("Hybrid 2018 prediction completed.")

print(
    pd.Series(
        hybrid_2018_pred
    ).value_counts()
)

Hybrid 2018 prediction completed.
0    6659427
1        105
Name: count, dtype: int64


In [18]:
# ==========================================
# EXPERIMENT 3 EVALUATION
# ==========================================

models_2018 = {
    "Random Forest": (
        rf_2018_pred,
        rf_2018_prob
    ),

    "Isolation Forest": (
        (
            iso_2018_norm >= BEST_THRESHOLD
        ).astype(int),
        iso_2018_norm
    ),

    "Hybrid": (
        hybrid_2018_pred,
        hybrid_2018_score
    )
}

results_exp3 = []

for name, (pred, score) in models_2018.items():

    results_exp3.append({

        "Model": name,

        "Precision": precision_score(
            y_2018,
            pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_2018,
            pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_2018,
            pred,
            zero_division=0
        ),

        "PR-AUC": average_precision_score(
            y_2018,
            score
        )
    })

results_exp3 = pd.DataFrame(
    results_exp3
)

results_exp3

,Model,Precision,Recall,F1,PR-AUC
0,Random Forest,0.330882,0.000068,0.000135,0.191616
1,Isolation Forest,0.056210,0.015814,0.024684,0.182102
2,Hybrid,0.990476,0.000078,0.000156,0.177846


In [19]:
# ==========================================
# EXPERIMENT 3 - CONFUSION MATRICES
# ==========================================

for name, (pred, score) in models_2018.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    cm = confusion_matrix(
        y_2018,
        pred,
        labels=[0, 1]
    )

    print(cm)


Random Forest
[[5328826     182]
 [1330434      90]]

Isolation Forest
[[4975718  353290]
 [1309483   21041]]

Hybrid
[[5329007       1]
 [1330420     104]]


In [20]:
# ==========================================
# EXPERIMENT 3 - CLASSIFICATION REPORTS
# ==========================================

for name, (pred, score) in models_2018.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(
        classification_report(
            y_2018,
            pred,
            labels=[0, 1],
            target_names=[
                "Benign",
                "Attack"
            ],
            zero_division=0
        )
    )


Random Forest
              precision    recall  f1-score   support

      Benign       0.80      1.00      0.89   5329008
      Attack       0.33      0.00      0.00   1330524

    accuracy                           0.80   6659532
   macro avg       0.57      0.50      0.44   6659532
weighted avg       0.71      0.80      0.71   6659532


Isolation Forest
              precision    recall  f1-score   support

      Benign       0.79      0.93      0.86   5329008
      Attack       0.06      0.02      0.02   1330524

    accuracy                           0.75   6659532
   macro avg       0.42      0.47      0.44   6659532
weighted avg       0.64      0.75      0.69   6659532


Hybrid
              precision    recall  f1-score   support

      Benign       0.80      1.00      0.89   5329008
      Attack       0.99      0.00      0.00   1330524

    accuracy                           0.80   6659532
   macro avg       0.90      0.50      0.44   6659532
weighted avg       0.84      0.80

In [21]:
# ==========================================
# SAVE EXPERIMENT 3 RESULTS
# ==========================================

EXP3_PATH = "exp_3_result"

os.makedirs(
    EXP3_PATH,
    exist_ok=True
)

results_exp3.to_csv(
    os.path.join(
        EXP3_PATH,
        "experiment_3_2018_results.csv"
    ),
    index=False
)

print("Experiment 3 results saved.")

print("\nSaved files:")

for file in os.listdir(EXP3_PATH):
    print(file)

Experiment 3 results saved.

Saved files:
experiment_3_2018_results.csv


In [22]:
# ==========================================
# LOAD EXPERIMENT 2 RESULTS
# ==========================================

EXP2_PATH = "exp_2_result"

results_exp2 = pd.read_csv(
    os.path.join(
        EXP2_PATH,
        "experiment_2_results.csv"
    )
)

print("Experiment 2 Results:")
display(results_exp2)

Experiment 2 Results:


,Model,Precision,Recall,F1,PR-AUC,FPR,FNR
0,Random Forest,1.0,0.190476,0.320000,1.0,0.0,0.809524
1,Isolation Forest,0.0,0.000000,0.000000,1.0,0.0,1.000000
2,Hybrid,1.0,0.047619,0.090909,1.0,0.0,0.952381


In [23]:
# ==========================================
# EXPERIMENT 2 vs EXPERIMENT 3
# ==========================================

comparison_exp2_exp3 = results_exp2.copy()

comparison_exp2_exp3["Experiment"] = "Experiment 2"

exp3_compare = results_exp3.copy()

exp3_compare["Experiment"] = "Experiment 3"

comparison_all = pd.concat(
    [
        comparison_exp2_exp3,
        exp3_compare
    ],
    ignore_index=True
)

print("=" * 70)
print("EXPERIMENT 2 vs EXPERIMENT 3")
print("=" * 70)

display(comparison_all)

EXPERIMENT 2 vs EXPERIMENT 3


,Model,Precision,Recall,F1,PR-AUC,FPR,FNR,Experiment
0,Random Forest,1.000000,0.190476,0.320000,1.000000,0.0,0.809524,Experiment 2
1,Isolation Forest,0.000000,0.000000,0.000000,1.000000,0.0,1.000000,Experiment 2
2,Hybrid,1.000000,0.047619,0.090909,1.000000,0.0,0.952381,Experiment 2
3,Random Forest,0.330882,0.000068,0.000135,0.191616,NaN,NaN,Experiment 3
4,Isolation Forest,0.056210,0.015814,0.024684,0.182102,NaN,NaN,Experiment 3
5,Hybrid,0.990476,0.000078,0.000156,0.177846,NaN,NaN,Experiment 3


In [24]:
# ==========================================
# MODEL-WISE COMPARISON
# ==========================================

model_comparison = comparison_all.pivot(
    index="Model",
    columns="Experiment",
    values=[
        "Precision",
        "Recall",
        "F1",
        "PR-AUC"
    ]
)

display(model_comparison)

Precision                    Recall               \
Experiment       Experiment 2 Experiment 3 Experiment 2 Experiment 3   
Model                                                                  
Hybrid                    1.0     0.990476     0.047619     0.000078   
Isolation Forest          0.0     0.056210     0.000000     0.015814   
Random Forest             1.0     0.330882     0.190476     0.000068   

                           F1                    PR-AUC               
Experiment       Experiment 2 Experiment 3 Experiment 2 Experiment 3  
Model                                                                 
Hybrid               0.090909     0.000156          1.0     0.177846  
Isolation Forest     0.000000     0.024684          1.0     0.182102  
Random Forest        0.320000     0.000135          1.0     0.191616

In [25]:
# ==========================================
# PERFORMANCE CHANGE
# ==========================================

metrics = [
    "Precision",
    "Recall",
    "F1",
    "PR-AUC"
]

exp2 = (
    results_exp2
    .set_index("Model")
)

exp3 = (
    results_exp3
    .set_index("Model")
)

performance_change = pd.DataFrame(
    index=exp2.index
)

for metric in metrics:

    performance_change[
        metric + "_Change"
    ] = (
        exp3[metric]
        -
        exp2[metric]
    )

print("=" * 70)
print("PERFORMANCE CHANGE: EXPERIMENT 3 - EXPERIMENT 2")
print("=" * 70)

display(performance_change)

PERFORMANCE CHANGE: EXPERIMENT 3 - EXPERIMENT 2


,Precision_Change,Recall_Change,F1_Change,PR-AUC_Change
Model,,,,
Random Forest,-0.669118,-0.190409,-0.319865,-0.808384
Isolation Forest,0.056210,0.015814,0.024684,-0.817898
Hybrid,-0.009524,-0.047541,-0.090753,-0.822154


In [26]:
# ==========================================
# BEST MODEL ON EXTERNAL 2018 DATA
# ==========================================

best_model_2018 = results_exp3.loc[
    results_exp3["F1"].idxmax()
]

print("=" * 60)
print("BEST MODEL ON CSE-CIC-IDS2018")
print("=" * 60)

print(
    "Model:",
    best_model_2018["Model"]
)

print(
    "Precision:",
    round(best_model_2018["Precision"], 4)
)

print(
    "Recall:",
    round(best_model_2018["Recall"], 4)
)

print(
    "F1:",
    round(best_model_2018["F1"], 4)
)

print(
    "PR-AUC:",
    round(best_model_2018["PR-AUC"], 4)
)

BEST MODEL ON CSE-CIC-IDS2018
Model: Isolation Forest
Precision: 0.0562
Recall: 0.0158
F1: 0.0247
PR-AUC: 0.1821


In [27]:
# ==========================================
# SAVE EXPERIMENT 3 COMPARISON
# ==========================================

comparison_all.to_csv(
    os.path.join(
        EXP3_PATH,
        "experiment_2_vs_experiment_3.csv"
    ),
    index=False
)

performance_change.to_csv(
    os.path.join(
        EXP3_PATH,
        "experiment_3_performance_change.csv"
    )
)

print("Comparison files saved successfully.")

print("\nFiles in Experiment 3 folder:")

for file in os.listdir(EXP3_PATH):
    print(file)

Comparison files saved successfully.

Files in Experiment 3 folder:
experiment_2_vs_experiment_3.csv
experiment_3_2018_results.csv
experiment_3_performance_change.csv
